In [23]:
import numpy as np
import matplotlib.pyplot as plt
ATU_navy =  "#001A79"
ATU_orange = "#FF791E"
ATU_green = "#005B5E"
ATU_teal = "#7BB9CB"

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=[ATU_navy])
plt.rcParams.update({
'text.color': ATU_navy,
'axes.labelcolor': ATU_navy,
'axes.titlecolor': ATU_navy,

'xtick.color': ATU_navy,
'ytick.color':ATU_navy,
})

from ipywidgets import interact, FloatSlider, IntSlider


# Fixed duration
duration = 0.5


def quantization_demo(
        peak_amplitude=4.0,
        frequency=10.0,
        adc_peak=5.0,
        bits=3,
        sampling_rate=1000,
        zoom=0.1):

    # =====================================================
    # Signal Generation
    # =====================================================
    t = np.arange(0, duration, 1 / sampling_rate)

    signal = peak_amplitude * np.sin(
        2 * np.pi * frequency * t
    )

    # =====================================================
    # ADC Parameters
    # =====================================================
    v_min = -adc_peak
    v_max = adc_peak

    levels = 2 ** bits

    lsb = (v_max - v_min) / (levels - 1)

    # Quantization voltage levels
    quant_levels = np.linspace(
        v_min,
        v_max,
        levels
    )

    # =====================================================
    # Clipping
    # =====================================================
    clipped_signal = np.clip(
        signal,
        v_min,
        v_max
    )

    # =====================================================
    # Quantization
    # =====================================================
    quantized_signal = (
        np.round(
            (clipped_signal - v_min) / lsb
        ) * lsb + v_min
    )

    # =====================================================
    # Error
    # =====================================================
    error = signal - quantized_signal

    # =====================================================
    # Clipping Statistics
    # =====================================================
    clipped_samples = np.sum(
        np.abs(signal) > adc_peak
    )

    clipping_percent = (
        100 * clipped_samples / len(signal)
    )

    # =====================================================
    # SQNR
    # =====================================================
    signal_power = np.mean(signal ** 2)
    noise_power = np.mean(error ** 2)

    if noise_power > 0:
        sqnr_measured = (
            10 * np.log10(signal_power / noise_power)
        )
    else:
        sqnr_measured = np.inf

    # =====================================================
    # SQNR Theory
    # =====================================================
    bits_range = np.arange(2, 13)
    sqnr_theoretical = 6.02 * bits_range + 1.76

    # =====================================================
    # Figure
    # =====================================================
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(12, 10),
        constrained_layout=True
    )

    # =====================================================
    # Plot 1 - Signal and Quantized Signal
    # =====================================================
    axes[0].plot(
        t,
        signal,
        linewidth=2,
        color=ATU_navy,
        label='Original Signal'
    )

    axes[0].plot(
        t,
        signal,
        'ro',
        markersize=4,
        alpha=0.5,
        label='Samples'
    )

    axes[0].step(
        t,
        quantized_signal,
        where='mid',
        color=ATU_orange,
        linewidth=1.5,
        label='Quantized Signal'
    )

    # ADC limits
    axes[0].axhline(
        adc_peak,
        color='black',
        linestyle='--',
        label='ADC Limits'
    )

    axes[0].axhline(
        -adc_peak,
        color='black',
        linestyle='--'
    )

    # Draw quantization levels
    for level in quant_levels:
        axes[0].axhline(
            level,
            color='gray',
            linestyle=':',
            linewidth=0.5,
            alpha=0.4
        )

    axes[0].set_title(
        'Signal, Sampling and Quantization'
    )

    axes[0].set_ylabel(
        'Voltage (V)'
    )
    
    axes[0].set_xlabel(
        'Time (s)'
    )


    axes[0].grid(True)

    axes[0].set_xlim(
        0,
        zoom
    )

    axes[0].set_ylim(
        -1.1 * adc_peak,
        1.1 * adc_peak
    )

    axes[0].legend()

    # =====================================================
    # Right-Hand ADC Code Axis
    # =====================================================
    ax_right = axes[0].twinx()

    ax_right.set_ylim(
        axes[0].get_ylim()
    )

    ax_right.set_yticks(
        quant_levels
    )

    ax_right.set_yticklabels(
        [str(i) for i in range(levels)]
    )

    ax_right.set_ylabel(
        'ADC Code'
    )

    # =====================================================
    # Plot 2 - Error
    # =====================================================
    axes[1].plot(
        t,
        error,
        color=ATU_green,
        linewidth=2
    )

    axes[1].plot(
        t,
        error,
        'go',
        markersize=3
    )

    axes[1].axhline(
        lsb / 2,
        color='black',
        linestyle='--',
        alpha=0.5
    )

    axes[1].axhline(
        -lsb / 2,
        color='black',
        linestyle='--',
        alpha=0.5
    )

    axes[1].set_title(
        'Quantization Error'
    )

    axes[1].set_xlabel(
        'Time (s)'
    )

    axes[1].set_ylabel(
        'Error (V)'
    )

    axes[1].set_xlim(
        0,
        zoom
    )

    axes[1].grid(True)

    # =====================================================
    # Plot 3 - SQNR
    # =====================================================
    axes[2].plot(
        bits_range,
        sqnr_theoretical,
        'o-',
        linewidth=2,
        label='Theoretical SQNR'
    )

    axes[2].scatter(
        bits,
        sqnr_measured,
        color=ATU_teal,
        s=120,
        zorder=10,
        label=f'Measured = {sqnr_measured:.2f} dB'
    )

    axes[2].set_title(
        'SQNR vs ADC Resolution'
    )

    axes[2].set_xlabel(
        'ADC Bits'
    )

    axes[2].set_ylabel(
        'SQNR (dB)'
    )

    axes[2].grid(True)
    axes[2].legend()

    # =====================================================
    # Summary
    # =====================================================
    max_error = np.max(
        np.abs(error)
    )

    fig.suptitle(
        f'Input Peak={peak_amplitude:.1f}V | '
        f'ADC Peak={adc_peak:.1f}V | '
        f'Bits={bits} | '
        f'Fs={sampling_rate}Hz | '
        f'LSB={lsb:.4f}V | '
        f'Max Error={max_error:.4f}V | '
        f'SQNR={sqnr_measured:.2f}dB | '
        f'Clipping={clipping_percent:.1f}%',
        fontsize=12
    )

    plt.show()


# ==========================================================
# Interactive Controls
# ==========================================================
interact(
    quantization_demo,

    peak_amplitude=FloatSlider(
        value=4.0,
        min=1.0,
        max=10.0,
        step=0.5,
        description='Signal Peak'
    ),

    frequency=FloatSlider(
        value=10.0,
        min=1.0,
        max=100.0,
        step=1.0,
        description='Freq (Hz)'
    ),

    adc_peak=FloatSlider(
        value=5.0,
        min=1.0,
        max=10.0,
        step=0.5,
        description='ADC Peak'
    ),

    bits=IntSlider(
        value=3,
        min=2,
        max=12,
        step=1,
        description='ADC Bits'
    ),

    sampling_rate=IntSlider(
        value=1000,
        min=50,
        max=5000,
        step=50,
        description='Fs (Hz)'
    ),

    zoom=FloatSlider(
        value=0.1,
        min=0.01,
        max=0.5,
        step=0.01,
        description='Zoom (s)'
    )
);

interactive(children=(FloatSlider(value=4.0, description='Signal Peak', max=10.0, min=1.0, step=0.5), FloatSli…